# Manufacturing Stress Adaptive-Agent Workbench

A companion to `manufacturing_stress_hybrid_workbench` for inspecting the persistent adaptive strategy, reviewing durable mutations, and generating executive-ready adaptation reports. This notebook intentionally does not run forecasts, backtests, LLM calls, or strategy mutations.

In [8]:
import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from dotenv import load_dotenv
from IPython.display import display

REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "uv.lock").exists() and (p / "implementations").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Repository root not found.")
for package_root in (REPO_ROOT / "aieng-forecasting", REPO_ROOT / "implementations"):
    if str(package_root) not in sys.path:
        sys.path.insert(0, str(package_root))
load_dotenv(REPO_ROOT / ".env", override=False)

RUN_ADAPTIVE_AGENT = True
RUN_ADAPTIVE_BACKTEST = True
LIVE_ADAPTATION = False
REFRESH_INPUT_DATA = False
AS_OF = "2017-01-01"
BACKTEST_STRIDE = 3
FORCE_REFRESH_AGENT_CACHE = False
STRATEGY_DIR = REPO_ROOT / "implementations" / "manufacturing_stress_forecasting" / "adaptive_agent" / "skills" / "manufacturing-strategy"
REPORT_DIR = REPO_ROOT / "implementations" / "manufacturing_stress_forecasting" / "reports" / "adaptive_agent_workbench"
STORE_DIR = REPO_ROOT / "data" / "predictions"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print({"run_adaptive_agent": RUN_ADAPTIVE_AGENT, "run_adaptive_backtest": RUN_ADAPTIVE_BACKTEST, "live_adaptation": LIVE_ADAPTATION, "as_of": AS_OF})

{'run_adaptive_agent': True, 'run_adaptive_backtest': True, 'live_adaptation': False, 'as_of': '2017-01-01'}


In [2]:
# Cell 0: synchronize the repository environment before running the workbench.
import subprocess
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Find the workspace root without confusing member pyproject files for it."""
    for candidate in (start, *start.parents):
        if (
            (candidate / "uv.lock").exists()
            and (candidate / "implementations").is_dir()
            and (candidate / "aieng-forecasting").is_dir()
        ):
            return candidate
    raise FileNotFoundError("Could not find the agentic-forecasting repository root.")


SETUP_ROOT = find_repo_root(Path.cwd().resolve())
print("Running: uv sync --all-extras --dev --all-packages")
setup_result = subprocess.run(
    ["uv", "sync", "--all-extras", "--dev", "--all-packages"],
    cwd=SETUP_ROOT,
    check=False,
    text=True,
    capture_output=True,
)
print(setup_result.stdout)
if setup_result.returncode != 0:
    print(setup_result.stderr)
    raise RuntimeError("uv sync failed. Install uv or restart the notebook with the repository environment selected.")
print("Environment synchronized successfully.")
print("If packages changed, restart the notebook kernel before continuing.")

Running: uv sync --all-extras --dev --all-packages

Environment synchronized successfully.
If packages changed, restart the notebook kernel before continuing.


## Load data, task specification, and strategy state

The notebook uses the same cutoff-aware manufacturing service and smoke task specification as the hybrid workbench. Adaptive execution is opt-in.

In [4]:
from aieng.forecasting.evaluation import BacktestSpec
from manufacturing_stress_forecasting.data import build_manufacturing_stress_service
from manufacturing_stress_forecasting.adaptive_agent.state import ManufacturingStrategyState

SPEC_PATH = REPO_ROOT / "implementations" / "manufacturing_stress_forecasting" / "specs" / "manufacturing_stress_smoke.yaml"
with SPEC_PATH.open(encoding="utf-8") as file:
    base_spec = BacktestSpec.model_validate(yaml.safe_load(file))
spec = base_spec.model_copy(update={"stride": BACKTEST_STRIDE})
service = build_manufacturing_stress_service(cache_dir=REPO_ROOT / "data" / "fred", yahoo_cache_dir=REPO_ROOT / "data" / "yfinance", refresh=REFRESH_INPUT_DATA)
context = service.context(as_of=pd.Timestamp(AS_OF).to_pydatetime())
state_path = STRATEGY_DIR / "skill_state.yaml"
state = ManufacturingStrategyState.model_validate(yaml.safe_load(state_path.read_text(encoding="utf-8")))
state_json = json.dumps(state.model_dump(mode="json"), sort_keys=True, separators=(",", ":"))
state_hash = hashlib.sha256(state_json.encode()).hexdigest()
print({"state_hash": state_hash, "strategy_dir": str(STRATEGY_DIR), "target": spec.task.target_series_id})
display(pd.DataFrame([{"status": h.status, "claim": h.claim} for h in state.hypotheses]))

{'state_hash': 'ee7d4b090d8898fca705b884e399d1a7df6d8cf6d3e3814f019b42be2f898497', 'strategy_dir': '/home/coder/BMO-C2-agentic-forecasting-copilot-push-to-bmo-c2/implementations/manufacturing_stress_forecasting/adaptive_agent/skills/manufacturing-strategy', 'target': 'manufacturing_stress'}


,status,claim
0,confirmed,Manufacturing stress is unlikely in the three-...


## Optional adaptive-agent execution

Set `RUN_ADAPTIVE_AGENT = True` to execute the stateful agent. The agent is built with its durable strategy mutation tools; leave the toggle false for inspection-only runs.

In [5]:
if RUN_ADAPTIVE_AGENT:
    from manufacturing_stress_forecasting.adaptive_agent.agent import build_manufacturing_adaptive_agent_predictor
    adaptive_predictor = build_manufacturing_adaptive_agent_predictor(strategy_dir=STRATEGY_DIR)
    adaptive_result = adaptive_predictor.predict(spec.task, context)[0]
    print(adaptive_result)
else:
    print("Adaptive-agent execution skipped; inspection-only mode is active.")

/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


predictor_id='agent_predictor_manufacturing_stress_adaptive_analyst_gemini_3_1_flash_lite_preview_gemini-3.1-flash-lite-preview_discrete' task_id='manufacturing_stress_3m' issued_at=datetime.datetime(2026, 10, 5, 3, 8, 2, 599871) as_of=datetime.datetime(2017, 1, 1, 0, 0) forecast_date=datetime.datetime(2017, 4, 1, 0, 0) payload=BinaryForecast(probability=0.06) metadata={'rationale': 'The 3-month IPMAN contraction of -0.94% remains well above the -2% stress threshold. Recent monthly stabilization and stable macro indicators (yield and credit spreads) support a slightly lower probability than the statistical anchor.', 'supporting_evidence': ['Current 3-month trailing IPMAN contraction of -0.94% is significantly above the -2% stress threshold.'], 'countervailing_evidence': ['None are material.'], 'direction': 'down', 'langfuse_trace_id': '5c53803b71b1b69a023998f05ce03f8b', 'langfuse_trace_url': 'https://us.cloud.langfuse.com/project/cmqqnar8v0eboad0cc5f3qinp/traces/5c53803b71b1b69a023998f

## Optional adaptive backtest

This uses the existing cached backtest runner and evaluates the adaptive agent across comparable origins. It remains disabled by default because it can issue multiple LLM calls and persist mutation events.

In [9]:
if RUN_ADAPTIVE_BACKTEST:
    from manufacturing_stress_forecasting.adaptive_agent.agent import build_manufacturing_adaptive_agent_predictor
    from manufacturing_stress_forecasting.hybrid_agent import HybridAgentPredictor
    from manufacturing_stress_forecasting.run_agent_backtest import run_or_load_backtest
    adaptive_backtest_predictor = HybridAgentPredictor(
        build_manufacturing_adaptive_agent_predictor(strategy_dir=STRATEGY_DIR),
        predictor_id="manufacturing_stress_adaptive_agent_v1",
    )
    adaptive_backtest_result = run_or_load_backtest(
        adaptive_backtest_predictor,
        spec,
        service,
        force_refresh=FORCE_REFRESH_AGENT_CACHE or REFRESH_INPUT_DATA,
        store_dir=STORE_DIR,
    )
    print({"mean_score": adaptive_backtest_result.mean_score, "metric": adaptive_backtest_result.metric, "scored_origins": len(adaptive_backtest_result.scores)})
else:
    print("Adaptive backtest skipped; inspection-only mode is active.")

KeyboardInterrupt: 

## Adaptation audit and executive report

In [7]:
from manufacturing_stress_forecasting.adaptive_agent.report import load_audit, write_report

AUDIT_PATH = STRATEGY_DIR / ".history" / "adaptation_audit.jsonl"
audit = load_audit(AUDIT_PATH) if AUDIT_PATH.exists() else pd.DataFrame()
if audit.empty:
    print(f"No durable adaptation mutations recorded at {AUDIT_PATH}.")
else:
    display(audit)
    paths = write_report(AUDIT_PATH, REPORT_DIR)
    print([path.name for path in paths])

,timestamp,event,tool,strategy_path,strategy_version,result,state_before_sha256,state_after_sha256
0,2026-10-05 02:54:07.456436+00:00,strategy_mutation,open_hypothesis,manufacturing-strategy,5ca0f5e2e3e9,Opened hyp-001; 3 confirmations are required b...,3118aab6b065cc0c5339d828859e9c46c99fe51b6f7f27...,5ca0f5e2e3e91523872ec8861390931044e6d178a6fd37...
1,2026-10-05 02:54:08.621283+00:00,strategy_mutation,record_observation,manufacturing-strategy,7fdc705e7171,Manufacturing observation recorded.,5ca0f5e2e3e91523872ec8861390931044e6d178a6fd37...,7fdc705e71712f8a5ca0344cc1a5316521bbbab6f49e47...
2,2026-10-05 02:54:09.462313+00:00,strategy_mutation,record_observation,manufacturing-strategy,89ab0cd60924,Manufacturing observation recorded.,7fdc705e71712f8a5ca0344cc1a5316521bbbab6f49e47...,89ab0cd60924f10c4fba9a874bc544131a5dee5432c561...
3,2026-10-05 02:54:10.266227+00:00,strategy_mutation,record_observation,manufacturing-strategy,aba23959efdb,Manufacturing observation recorded.,89ab0cd60924f10c4fba9a874bc544131a5dee5432c561...,aba23959efdb5dc1dafc7e8c243f039ee458a725004a99...
4,2026-10-05 02:54:13.757865+00:00,strategy_mutation,record_hypothesis_outcome,manufacturing-strategy,8ce7d315ce31,Recorded confirmed outcome for hyp-001.,aba23959efdb5dc1dafc7e8c243f039ee458a725004a99...,8ce7d315ce31f829da8c04afa2accc84b05cbd951928bc...
5,2026-10-05 02:54:14.363148+00:00,strategy_mutation,record_hypothesis_outcome,manufacturing-strategy,8d90e2d77d9b,Recorded confirmed outcome for hyp-001.,8ce7d315ce31f829da8c04afa2accc84b05cbd951928bc...,8d90e2d77d9b0516845227849de75069c993bfa6da9a5d...
6,2026-10-05 02:54:15.129828+00:00,strategy_mutation,record_hypothesis_outcome,manufacturing-strategy,48997251b9f0,Recorded confirmed outcome for hyp-001.,8d90e2d77d9b0516845227849de75069c993bfa6da9a5d...,48997251b9f02b73386e9e2a426f2c2901161db838612b...
7,2026-10-05 02:54:16.158524+00:00,strategy_mutation,graduate_hypothesis,manufacturing-strategy,609d8b85ec48,Graduated hyp-001 into a calibration correction.,48997251b9f02b73386e9e2a426f2c2901161db838612b...,609d8b85ec48e191b01d20713b40aa6f5cf7c8d717d737...
8,2026-10-05 02:54:17.079875+00:00,strategy_mutation,update_approach_narrative,manufacturing-strategy,ee7d4b090d88,Manufacturing approach narrative updated.,609d8b85ec48e191b01d20713b40aa6f5cf7c8d717d737...,ee7d4b090d8898fca705b884e399d1a7df6d8cf6d3e381...


['adaptation_timeline.png', 'mutation_activity.png']


## Interpretation checklist

- Compare inspection and execution runs using the strategy hash, trace metadata, and mutation audit.
- Treat adaptation as durable only when the redacted JSONL audit and post-state hash are present.
- Review mutation activity alongside forecast quality; mutation count alone is not evidence of improvement.
- Keep `RUN_ADAPTIVE_AGENT = False` for reproducible inspection-only runs.